# 00 — Data and protocol

**Research question:** What are the experiment inputs and definitions?

[Full input manifest](../output/paper_v1/input_manifest.csv) · [Numerical thresholds](../output/paper_v1/pilot_parameters.csv) · [Status and gaps](../output/paper_v1/status.md)

## Inputs and definitions
Retain the first and last input vertices. Outputs are original-vertex subsequences and may have unequal lengths. Minimize $k=\max(|A'|,|B'|)$. Discrete couplings may repeat an index. Auxiliary points represent matching positions, never selectable output vertices.

| Method | Input fidelity | Output coupling |
|---|---|---|
| Independent continuous | global continuous Fréchet | measured, unconstrained |
| Independent discrete | discrete Fréchet | measured, unconstrained |
| CPS-2F | global continuous Fréchet | discrete, at most δ₃ |
| CPS-3F | discrete Fréchet | discrete, at most δ₃ |

For each pair, $s_A,s_B$ are the mean input edge lengths; $\alpha\in\{0.5,1\}$, $\delta_1=\alpha s_A$, $\delta_2=\alpha s_B$, and $\delta_3=d_{dF}(A,B)$ without rounding. All four methods receive identical inputs and thresholds. This is a pipeline pilot, not final paper parameter selection.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks_paper': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks_paper'))
from paper_core import OUT, PROTOCOL, METHODS, compact_results, plot_domain
import pandas as pd
from IPython.display import display, Markdown
# Central parameters: frozen before optimization; no notebook-specific overrides.
PARAMETERS = json.loads((OUT / 'protocol.json').read_text())
assert PARAMETERS['alphas'] == [0.5, 1.0]


In [ ]:
inventory = pd.read_csv(OUT / 'existing_inventory.csv')
print('Existing artifacts inventoried:', len(inventory))
display(inventory.groupby('kind').size().rename('count').to_frame())
display(pd.read_csv(OUT / 'prior_experiments.csv'))

## Frozen cohort, selected by size before methods ran
Two smallest eligible aligned protein pairs at the fixed, already supplied w=16 resolution; the smallest supplied hurricane pair in each basin. Ties use identifiers. This is not all available pairs. Selection uses no method outcomes. [Complete selection audit](../output/paper_v1/selection_audit.csv).

In [ ]:
manifest = pd.read_csv(OUT / 'input_manifest.csv')
display(manifest[['pair','domain','nA','nB','units','extent_policy']])
brief = manifest[['pair','subsampling','gaps_A','gaps_B','alignment']].copy()
brief['subsampling'] = brief['subsampling'].str.split(';').str[0]
display(brief)
selection = pd.read_csv(OUT / 'selection_audit.csv')
display(selection.groupby('domain').agg(eligible_pairs=('pair','size'), selected_pairs=('selected','sum')))
display(pd.read_csv(OUT / 'shared_entities.csv'))

Both protein pairs reuse 1o7j.a and are statistically dependent. They are entire supplied **subsampled** curves, not full-resolution backbones. Protein gaps and prior alignment remain as supplied. Hurricanes retain the common coordinate frame within each basin; no new fit or subsampling is performed. Coordinates have no timestamps, so no temporal synchronization is inferred.

In [ ]:
parameters = pd.read_csv(OUT / 'pilot_parameters.csv')
with pd.option_context('display.precision', 9):
    display(parameters[['pair','nA','nB','alpha','s_A','s_B','delta1','delta2','delta3','units']])
print('Configurations:', len(parameters) * len(METHODS))
print('Limits:', PARAMETERS['solver_limits'], 'hard seconds:', PARAMETERS['hard_seconds'], 'MiB:', PARAMETERS['memory_mib'])

## Provenance and interpretation
[Inventory with hashes](../output/paper_v1/existing_inventory.csv), [reference document hashes](../output/paper_v1/references/manifest.json), and [frozen protocol](../output/paper_v1/protocol.json) provide traceability. The reference definitions and configuration graph are used as specifications. Thesis experiment numbers and conclusions are not reused as evidence. An output contains one more vertex than its number of index advances; validation explicitly checks this bookkeeping.

One execution per configuration, after worker warm-up; runtimes are diagnostic. A resource exit does not establish infeasibility. Certificate routes are reported separately from configuration-graph execution. Later graph-cost and paper-summary stages are planned only.

## Supported conclusions and remaining gaps
The cohort and thresholds are reproducible. Its small size, coarse protein sampling, shared structure, and absent hurricane timestamps limit generalization. See status.md for completed versus missing work.

**Upstream protein coverage caveat:** The existing identifier audit records 325 supplied Cα coordinates for 1d9q.d versus a reported paper length of 297, and three contiguous fragments before representing the stored chain as a single curve. This pilot retains the supplied gap-bridging edges and applies no new fragmentation. That audit is provenance metadata, not a new structural verification in Stage 1. “Entire supplied curve” does not mean a complete biological polymer.